# Building a Todo Application with Streamlit, FastAPI, and Firebase

This notebook documents the process of building a Todo application using Streamlit for the frontend, FastAPI for the backend, Firebase Authentication for user login, and Cloud Firestore for persistent storage.


## 1. Objectives

The application must support the core requirements below:

- Register and log in with email/password.
- Create, view, update, and delete Todo items.
- Only display Todo items that belong to the currently authenticated user.

Additional features implemented:

- Search Todo items by title or description.
- Filter Todo items by status and priority.
- Basic role-based access control with `user` and `admin` roles.
- Optional Google sign-in integration.


## 2. Folder Structure

```text
todo-app/
|-- backend/
|   |-- main.py
|   |-- requirements.txt
|   `-- app/
|       |-- config.py
|       |-- firebase.py
|       |-- auth.py
|       |-- schemas.py
|       `-- routes/
|           |-- users.py
|           `-- todos.py
|-- frontend/
|   |-- app.py
|   |-- config.py
|   |-- firebase_auth.py
|   |-- api_client.py
|   |-- session.py
|   |-- components.py
|   `-- requirements.txt
|-- .env.example
|-- .gitignore
|-- README.md
`-- procedure.ipynb
```

The code is split into small modules so each file has one main responsibility. This keeps the project easier to read, test, explain, and extend.


## 3. High-Level Architecture

The system has four main parts:

- **Streamlit frontend**: renders login forms, Todo forms, filters, and dashboard controls.
- **Firebase Authentication**: authenticates users and returns a Firebase ID token.
- **FastAPI backend**: verifies Firebase ID tokens, applies authorization rules, and exposes Todo REST endpoints.
- **Cloud Firestore**: stores users and Todo documents.

The frontend does not access Firestore directly. All Todo operations go through FastAPI so the backend can verify tokens and enforce ownership rules.


## 4. Authentication Flow

1. The user enters an email and password in the Streamlit sidebar.
2. Streamlit calls the Firebase Authentication REST API.
3. Firebase returns an `idToken` if the credentials are valid.
4. Streamlit stores the token and user metadata in `st.session_state`.
5. For every backend request, Streamlit sends the token in the HTTP header:

```http
Authorization: Bearer <Firebase idToken>
```

6. FastAPI verifies the token using Firebase Admin SDK.
7. If the token is valid, FastAPI extracts the user's `uid` and processes the request.


## 5. Firestore Data Design

Collection `users` stores role metadata. The document id is the Firebase `uid`.

```json
{
  "role": "user",
  "email_verified": false,
  "created_at": "2026-05-13T..."
}
```

Collection `todos` stores Todo items.

```json
{
  "title": "Study FastAPI",
  "description": "Finish the Todo assignment",
  "due_date": "2026-05-20",
  "priority": "normal",
  "done": false,
  "owner_uid": "firebase-user-uid",
  "created_at": "2026-05-13T...",
  "updated_at": "2026-05-13T..."
}
```

`owner_uid` is the key field for data isolation. Normal users can only access Todo documents where `owner_uid` matches their authenticated Firebase uid.


## 6. Backend Design

The backend uses FastAPI and Firebase Admin SDK.

- `backend/main.py` creates the FastAPI app and includes routers.
- `backend/app/config.py` reads environment variables.
- `backend/app/firebase.py` initializes Firebase Admin SDK and Firestore.
- `backend/app/auth.py` verifies Firebase ID tokens and creates a user document if needed.
- `backend/app/schemas.py` defines Pydantic models.
- `backend/app/routes/todos.py` contains Todo CRUD endpoints.
- `backend/app/routes/users.py` contains `/me` and admin role management.

Main endpoints:

- `GET /me`: return the current user's uid, email, and role.
- `GET /todos`: list Todo items with optional search and filters.
- `POST /todos`: create a Todo item for the current user.
- `PUT /todos/{todo_id}`: update an existing Todo item.
- `DELETE /todos/{todo_id}`: delete a Todo item.
- `POST /users/{target_uid}/role`: allow an admin to change another user's role.


## 7. Frontend Design

The frontend uses Streamlit.

- `frontend/app.py` is the entrypoint and controls page flow.
- `frontend/config.py` reads frontend environment variables.
- `frontend/firebase_auth.py` calls Firebase Authentication REST endpoints.
- `frontend/api_client.py` calls the FastAPI backend.
- `frontend/session.py` manages Streamlit authentication session state.
- `frontend/components.py` renders login, Todo dashboard, Todo editor, filters, and admin UI.

The frontend keeps the Firebase ID token in session state and attaches it to backend requests. It never uses the Firebase service account key.


## 8. Firebase Setup

Required Firebase steps:

1. Create a Firebase project.
2. Enable Email/Password sign-in in Firebase Authentication.
3. Optionally enable Google sign-in for the advanced feature.
4. Create a Cloud Firestore database.
5. Copy the Firebase Web API key for the frontend.
6. Create a service account JSON file for the backend.

The service account JSON is private and must never be committed to GitHub.


## 9. Environment Variables

Use `.env.example` as the template and create a local `.env` file.

Important variables:

- `GOOGLE_APPLICATION_CREDENTIALS`: path to the backend service account JSON file.
- `FIREBASE_PROJECT_ID`: Firebase project id.
- `BACKEND_CORS_ORIGINS`: allowed Streamlit origins for FastAPI CORS.
- `FIREBASE_API_KEY`: Firebase Web API key used by Firebase Authentication REST API.
- `API_BASE_URL`: FastAPI backend URL.
- `GOOGLE_OAUTH_CLIENT_ID`, `GOOGLE_OAUTH_CLIENT_SECRET`, `GOOGLE_REDIRECT_URI`: optional Google login settings.

For submission, include `.env.example` but do not include `.env` or real secret files.


## 10. Running Locally

Install and run the backend:

```bash
pip install -r backend/requirements.txt
uvicorn backend.main:app --reload --host 0.0.0.0 --port 8000
```

Install and run the frontend:

```bash
pip install -r frontend/requirements.txt
streamlit run frontend/app.py
```

Then open `http://localhost:8501`.


## 11. Basic Testing Checklist

Manual test cases:

1. Register user A with email/password.
2. Create multiple Todo items for user A.
3. Edit a Todo title, description, due date, and priority.
4. Toggle a Todo between Open and Done.
5. Search Todo items by keyword.
6. Filter Todo items by status and priority.
7. Delete a Todo item.
8. Register user B and confirm user B cannot see user A's Todo items.
9. Promote a user to admin and confirm the admin can review all Todo items.


## 12. Security and Submission Notes

The Firebase Web API key is not the same as the Firebase Admin SDK service account key. The Web API key is commonly present in frontend applications. However, the service account JSON contains private credentials and must not be submitted.

Do not submit:

- `.env`
- Firebase service account JSON
- Streamlit secrets file
- Any real OAuth client secret or private key

Recommended submission package:

- Source code
- `.env.example`
- `README.md`
- `procedure.ipynb`
- A deployed application link for grading/demo purposes
